In [ ]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati

#funzione modello
def model(x, par):
    p = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return B + (C*(p-A))/((p-A)-C)

def model_derivative(p, par):
    return -par[2]**2 / ((p - par[0]) - par[2])**2

#leggere file excel 
xlsx ="esperienza3.xlsx"
df = pd.read_excel("esperienza2.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["p", "q", "sigma_p", "sigma_q"])
df = df.to_numpy(dtype = 'float64')

#definizione variabili
p = df[:, 9] 
q = df[:, 10] 


err_p = df[:, 8]
err_q =  df[:, 7]


p = np.delete(p, [0, 62, 88] )
q = np.delete(q, [0, 62, 88]  )
err_p = np.delete(err_p, [0, 62, 88] )
err_q = np.delete(err_q, [0, 62, 88] )






#minimizzazzione con ROOT
model_fit = ROOT.TF1('model', model, min(p), max(p), 3)
model_fit.SetParameters(0.0, 0.0, 0.0)
model_fit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr1.SetTitle("Dati sperimentali e fit; p; q")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(model_fit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = model_fit.GetParameters()

errors = model_fit.GetParErrors()

chi2 = fit_res.Chi2()

err_qeff = np.sqrt(err_q**2 + ((model_derivative(p, par)) * err_p)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(model_fit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("fit_root.png")

IndexError: index 9 is out of bounds for axis 1 with size 9

## Fit separati a p = 400



In [ ]:

p_cut = 400

model_fit_1 = ROOT.TF1('model_1', model, min(p), p_cut, 3)      # parte con p < 400
model_fit_1.SetParameters(par[0], par[1], par[2])
model_fit_1.SetParNames("A", "B", "C")
model_fit_1.SetLineColor(ROOT.kRed + 1)
model_fit_1.SetLineWidth(2)

model_fit_2 = ROOT.TF1('model_2', model, p_cut, max(p), 3)      # parte con p > 400
model_fit_2.SetParameters(par[0], par[1], par[2])
model_fit_2.SetParNames("A", "B", "C")
model_fit_2.SetLineColor(ROOT.kGreen + 2)
model_fit_2.SetLineWidth(2)

gr_split = ROOT.TGraphErrors(len(p), p, q, err_p, err_q)
gr_split.SetTitle("Fit separati; p; q")
gr_split.SetMarkerStyle(20)
gr_split.SetMarkerSize(0.7)
gr_split.SetMarkerColor(ROOT.kAzure + 2)
gr_split.SetLineColor(ROOT.kAzure + 3)
gr_split.SetLineWidth(1)

fit_res_1 = gr_split.Fit(model_fit_1, "SRN")
fit_res_2 = gr_split.Fit(model_fit_2, "SRN")

par_1 = model_fit_1.GetParameters()
par_2 = model_fit_2.GetParameters()
errors_1 = model_fit_1.GetParErrors()
errors_2 = model_fit_2.GetParErrors()
chi2_1, ndf_1 = fit_res_1.Chi2(), fit_res_1.Ndf()
chi2_2, ndf_2 = fit_res_2.Chi2(), fit_res_2.Ndf()


print(f"Fit 1 (p < {p_cut}):")
print(f"A = {par_1[0]} ± {errors_1[0]}")
print(f"B = {par_1[1]} ± {errors_1[1]}")
print(f"C = {par_1[2]} ± {errors_1[2]}")
print(f"Chi2 = {chi2_1}   ndf = {ndf_1}")
print()
print(f"Fit 2 (p > {p_cut}):")
print(f"A = {par_2[0]} ± {errors_2[0]}")
print(f"B = {par_2[1]} ± {errors_2[1]}")
print(f"C = {par_2[2]} ± {errors_2[2]}")
print(f"Chi2 = {chi2_2}   ndf = {ndf_2}")

print()
for i, nome in enumerate(["A", "B", "C"]):
    diff = np.abs(par_1[i] - par_2[i])
    sigma = np.sqrt(errors_1[i]**2 + errors_2[i]**2)
    if diff < 3*sigma:
        print(f"{nome}: differenza = {diff:.3f} ({diff/sigma:.1f} sigma) -> compatibili")
    else:
        print(f"{nome}: differenza = {diff:.3f} ({diff/sigma:.1f} sigma) -> NON compatibili")

chi2_tot = model_fit.GetChisquare()      
ndf_tot = model_fit.GetNDF()
chi2_split = chi2_1 + chi2_2            
ndf_split = ndf_1 + ndf_2
print()
print(f"Fit unico    : chi2 = {chi2_tot:.1f}  ndf = {ndf_tot}  chi2/ndf = {chi2_tot/ndf_tot:.1f}")
print(f"Due fit somma: chi2 = {chi2_split:.1f}  ndf = {ndf_split}  chi2/ndf = {chi2_split/ndf_split:.1f}")

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      3864.73
NDf                       =           75
Edm                       =  1.16969e-05
NCalls                    =           94
A                         =     -7.76433   +/-   0.804223    
B                         =     -22.3676   +/-   1.1953      
C                         =      203.738   +/-   0.485386    
Fit 1 (p < 400):
A = -7.764330881768524 ± 0.8042225535865124
B = -22.3675585771755 ± 1.1952974562957177
C = 203.73781200309483 ± 0.4853859908602273
Chi2 = 3864.7312312248014   ndf = 75

Fit 2 (p > 400):
A = -9.334617962563101 ± 0.5998512822227496
B = -8.190189498967799 ± 0.30502728572738713
C = 202.50706664414724 ± 0.21005649021131234
Chi2 = 1819.2646334975868   ndf = 32

A: differenza = 1.570 (1.6 sigma) -> compatibili
B: differenza = 14.177 (11.5 sigma) -> NON compatibili
C: differenza = 1.231 (2.3 sigma) -> compatibili

Fit unico    : chi2 = 14473.8  ndf = 110  chi2/ndf

In [ ]:
c_split = ROOT.TCanvas("c_split", "Fit separati", 800, 600)
c_split.SetGrid()

gr_split.Draw("AP")
model_fit_1.Draw("same")
model_fit_2.Draw("same")

leg_split = ROOT.TLegend(0.40, 0.72, 0.89, 0.89)
leg_split.AddEntry(gr_split, "Dati", "pe")
leg_split.AddEntry(model_fit_1, f"Fit p < {p_cut}: A = {par_1[0]:.1f} #pm {errors_1[0]:.1f}, #chi^{{2}}/ndf = {chi2_1:.0f}/{ndf_1}", "l")
leg_split.AddEntry(model_fit_2, f"Fit p > {p_cut}: A = {par_2[0]:.1f} #pm {errors_2[0]:.1f}, #chi^{{2}}/ndf = {chi2_2:.0f}/{ndf_2}", "l")
leg_split.Draw()

c_split.Draw()
c_split.SaveAs("fit_root_split.png")

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_split
Info in <TCanvas::Print>: png file fit_root_split.png has been created


In [ ]:
err_R = 1
err_L = 1
err_x0 = 0.115470053837925

if np.abs(par[0]) < 3*(errors[0] + err_x0 + err_R + err_L):
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*(errors[1] + err_x0  + err_L):
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A NON è compatibile con 0
---
B NON è compatibile con 0


In [ ]:
q_attesi = np.array([model_fit.Eval(x_val) for x_val in p])
residuals_norm = (q - q_attesi) / err_qeff



#residuals_norm = np.delete(residuals_norm, [0,1, 10, 12, 14, 43, 44, 52])

#print(residuals_norm)

gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(q), max(q), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 20, min(residuals_norm), max(residuals_norm))
hist.Fill(residuals_norm)
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0.13, 0.80, 0.35, 0.89)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("hist.png")


print()
print()

for i in np.arange(0,len(residuals_norm)):
    if residuals_norm[i] < -0.8:
        print (residuals_norm[i])
        print(i)

Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).


Fit status: 0
Parametri stimati:
const = 63.96419757793367 ± 8.027487470467825
#mu = 4.67162260732875 ± 0.24838694559499647
#sigma = 2.000872776212769 ± 0.1354358200363518
Chi2 = 32.95106886746761


-1.560766466470515
66
-8.50869079887931
67
-4.415853489023314
68
-7.27727991329791
70
-14.757961332068515
71
-24.735388000467168
72
-31.54276035991086
73
-52.15635218184569
74
-63.543683458215085
75
-15.478637614291358
76
-9.824071144295981
91
-2.4339953490834576
96
-10.775043036226988
98
-4.304830833054854
102
-8.480404784532478
103
-4.3223962840501935
104
-10.867733714859703
106
-12.188793417881133
107
-6.510809323126725
108
-6.9525777370618576
110
-1.2273498123913982
111
-7.331735700828858
112
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      32.9511
NDf                       =           10
Edm                       =  6.86564e-07
NCalls                    =          160
const                     =      63.9642   +/-   8.02749     
mu

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file hist.png has been created


In [ ]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1]) < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if np.abs(parameters[2] - 1) < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const NON è compatibile con 1
---
mu NON è compatibile con 0
---
sigma NON è compatibile con 1


In [ ]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = q - q_attesi
print(residuals)

line_fit = ROOT.TF1('line', line, min(p), max(p), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(p), p, residuals, np.zeros_like(residuals), err_qeff)
gr2.SetTitle("Residui in funzione di p; p; q - q_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("APX")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("fit_line.png")

[ 1.24818840e+02  1.51312773e+02  1.17475784e+02  1.16418581e+02
  1.11557785e+02  1.08677305e+02  9.88858466e+01  9.98787974e+01
  8.79586819e+01  8.34073415e+01  9.20275446e+01  7.40507554e+01
  7.59107602e+01  6.77643615e+01  7.88080903e+01  7.67886053e+01
  7.28348621e+01  6.68302625e+01  6.58291904e+01  5.96406762e+01
  6.18467112e+01  5.79062915e+01  6.02332581e+01  5.40388076e+01
  5.08341619e+01  4.68082796e+01  4.37549272e+01  4.69160983e+01
  4.66340123e+01  4.25823316e+01  4.61864882e+01  4.19562601e+01
  4.03642374e+01  3.80693313e+01  3.66438593e+01  3.47338500e+01
  3.44963723e+01  3.24125830e+01  3.31579962e+01  2.78673045e+01
  2.94681492e+01  2.79367555e+01  2.23365703e+01  1.99970754e+01
  2.45462737e+01  2.14832051e+01  2.23027202e+01  2.39683918e+01
  2.15353982e+01  2.14864772e+01  2.33204738e+01  1.90950173e+01
  2.12101109e+01  1.97642741e+01  2.12091882e+01  1.98113608e+01
  1.95940350e+01  1.60283943e+01  1.55986125e+01  1.50153002e+01
  9.79819515e-01  2.22849

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file fit_line.png has been created
